# Threads archiver

This notebook creates an archive of the YouTube threads retrieved with PYG lean via the API.
The results are stored in a folder specified by the user below.

In [ ]:
import os
import zipfile
import json
from collections import defaultdict
from datetime import datetime

# Functions

In [1]:
# Step 1: Extract threadsdata
def extract_threadsdata(data):
    if isinstance(data, dict):
        data = list(data.values())[0] if data else []

    threadsdata = []
    for item in data:
        if not isinstance(item, dict):
            continue

        try:
            thread = {
                "id": item.get("id"),
                "snippet": {
                    "topLevelComment": {
                        "snippet": {
                            "textDisplay": item["snippet"]["topLevelComment"]["snippet"].get("textDisplay"),
                            "textOriginal": item["snippet"]["topLevelComment"]["snippet"].get("textOriginal"),
                            "authorDisplayName": item["snippet"]["topLevelComment"]["snippet"].get("authorDisplayName"),
                            "likeCount": item["snippet"]["topLevelComment"]["snippet"].get("likeCount", 0),
                            "publishedAt": item["snippet"]["topLevelComment"]["snippet"].get("publishedAt"),
                            "updatedAt": item["snippet"]["topLevelComment"]["snippet"].get("updatedAt")
                        }
                    },
                    "totalReplyCount": item["snippet"].get("totalReplyCount", 0)
                }
            }
            threadsdata.append(thread)
        except KeyError as e:
            print(f"KeyError encountered: {e}. Skipping item.")

    return threadsdata

# Step 2: Extract threads from ZIP files
def extract_threads_from_zips(zip_directory, extract_directory):
    threads_files = defaultdict(list)

    for zip_file in os.listdir(zip_directory):
        if zip_file.endswith('.zip'):
            zip_path = os.path.join(zip_directory, zip_file)
            zip_modification_time = os.path.getmtime(zip_path)

            zip_extract_folder = extract_directory
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                for file in zip_ref.namelist():
                    if file.endswith('_threads.json'):
                        zip_filename = os.path.splitext(zip_file)[0]
                        extracted_file_path = os.path.join(
                            zip_extract_folder,
                            f"{zip_filename}__{os.path.basename(file)}"
                        )
                        with open(extracted_file_path, 'wb') as f_out:
                            f_out.write(zip_ref.read(file))
                        
                        file_id = os.path.basename(file).split('__')[0]
                        threads_files[file_id].append((extracted_file_path, zip_modification_time))
    return threads_files

# Step 3: Compare and merge threadsdata
def compare_and_merge_threadsdata(threads_files):
    merged_results = {}

    for file_id, files in threads_files.items():
        # Sort files by modification time
        files.sort(key=lambda x: x[1])

        thread_data = defaultdict(list)

        for file_path, modification_time in files:
            timestamp = datetime.fromtimestamp(modification_time).isoformat()

            with open(file_path, 'r', encoding='utf-8') as f:
                current_data = json.load(f)
                current_threadsdata = extract_threadsdata(current_data)

            for thread in current_threadsdata:
                thread_id = thread["id"]

                # If this is the first record for this thread_id, save the complete snippet
                if not thread_data[thread_id]:
                    thread_data[thread_id].append({
                        "timestamp": timestamp,
                        "snippet": thread["snippet"]
                    })
                else:
                    # Get the last recorded data (last update or initial snippet)
                    last_entry = thread_data[thread_id][-1]

                    # Compare likeCount and totalReplyCount
                    current_like_count = thread["snippet"]["topLevelComment"]["snippet"]["likeCount"]
                    current_total_reply_count = thread["snippet"]["totalReplyCount"]

                    # Get the last recorded likeCount and totalReplyCount
                    last_like_count = (
                        last_entry.get("likeCount") if "likeCount" in last_entry
                        else last_entry["snippet"]["topLevelComment"]["snippet"]["likeCount"]
                    )
                    last_total_reply_count = (
                        last_entry.get("totalReplyCount") if "totalReplyCount" in last_entry
                        else last_entry["snippet"]["totalReplyCount"]
                    )

                    # Only add if there's a change and it's not a duplicate
                    if current_like_count != last_like_count or current_total_reply_count != last_total_reply_count:
                        thread_data[thread_id].append({
                            "likeCount": current_like_count,
                            "totalReplyCount": current_total_reply_count,
                            "timestamp": timestamp
                        })

        # Format the results for this file_id
        merged_results[file_id] = [
            {
                "id": thread_id,
                "updates": updates
            }
            for thread_id, updates in thread_data.items()
        ]

    return merged_results

# Step 4: Save merged threadsdata
def save_merged_threadsdata(merged_results, output_directory):
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)

    for file_id, data in merged_results.items():
        output_file = os.path.join(output_directory, f'{file_id}_threadsdata.json')
        with open(output_file, 'w', encoding='utf-8') as f_out:
            json.dump(data, f_out, ensure_ascii=False, indent=4)

In [ ]:
# Main function
def main(zip_directory, extract_directory, output_directory):
    threads_files = extract_threads_from_zips(zip_directory, extract_directory)
    merged_results = compare_and_merge_threadsdata(threads_files)
    save_merged_threadsdata(merged_results, output_directory)

# Execution Section

In [ ]:
# Replace with your actual directory paths
zip_directory = 'D:\\JPMETA'
extract_directory = 'D:\\JPMETA\\extracted'
output_directory = 'D:\\JPMETA\\thread'

main(zip_directory, extract_directory, output_directory)